# Behavioral Analysis: Agent Actions and Interactions

This notebook analyzes individual agent behavior patterns including
action diversity, interaction frequency, and temporal activity patterns.

In [ ]:
from data_loader import (
    load_experiment,
    build_agent_timeline,
    extract_action_events,
    extract_reflection_events,
    compute_interaction_matrix,
)

from collections import Counter, defaultdict
import json

## 1. Load Data

In [ ]:
experiment_id = "exp_abc123"  # <-- change this
events = load_experiment(experiment_id)
actions = extract_action_events(events)
reflections = extract_reflection_events(events)

agents = sorted(set(e.get("agent_id", "") for e in events if e.get("agent_id")))
print(f"Loaded {len(events)} events, {len(agents)} agents")

## 2. Action Frequency per Agent

In [ ]:
action_counts = Counter(e.get("agent_id") for e in actions)
print("Actions per agent:")
for agent, count in action_counts.most_common():
    print(f"  {agent}: {count}")

## 3. Action Diversity

How unique are each agent's actions? High diversity means varied behavior.

In [ ]:
agent_action_texts: dict[str, list[str]] = defaultdict(list)
for e in actions:
    aid = e.get("agent_id", "")
    data = e.get("data", {})
    if isinstance(data, dict):
        agent_action_texts[aid].append(data.get("action", ""))
    elif isinstance(data, str):
        agent_action_texts[aid].append(data)

print("Action diversity (unique / total):")
for agent in agents:
    texts = agent_action_texts.get(agent, [])
    unique = len(set(t.strip() for t in texts if t.strip()))
    total = len(texts)
    ratio = unique / total if total > 0 else 0
    print(f"  {agent}: {unique}/{total} ({ratio:.2f})")

## 4. Temporal Activity Pattern

When are agents most active? Group actions by simulation step.

In [ ]:
step_activity: dict[int, int] = Counter(e.get("step", 0) for e in actions)
steps_sorted = sorted(step_activity.keys())

if steps_sorted:
    print(f"Activity range: step {steps_sorted[0]} to {steps_sorted[-1]}")
    print(f"Average actions per step: {sum(step_activity.values()) / len(step_activity):.1f}")
    
    # Find peak activity steps
    peak_steps = step_activity.most_common(5)
    print("\nPeak activity steps:")
    for step, count in peak_steps:
        print(f"  Step {step}: {count} actions")

## 5. Interaction Matrix

Which agents interact with each other most frequently?

In [ ]:
interaction_matrix = compute_interaction_matrix(events)

print("Interaction frequency matrix:")
print(f"{'Agent':<15} | {'Interactions':>5} | Top partners")
print("-" * 60)
for agent in agents:
    partners = interaction_matrix.get(agent, {})
    total = sum(partners.values())
    top = sorted(partners.items(), key=lambda x: -x[1])[:3]
    top_str = ", ".join(f"{name}({cnt})" for name, cnt in top)
    print(f"{agent:<15} | {total:>5} | {top_str or 'none'}")

## 6. Reflection Analysis

Examine what agents reflect on and how often.

In [ ]:
reflection_counts = Counter(e.get("agent_id") for e in reflections)

print("Reflections per agent:")
for agent, count in reflection_counts.most_common():
    agent_refs = [e for e in reflections if e.get("agent_id") == agent]
    print(f"  {agent}: {count} reflections")
    if agent_refs:
        data = agent_refs[0].get("data", {})
        trigger = data.get("trigger", "unknown") if isinstance(data, dict) else "unknown"
        print(f"    First trigger: {trigger}")

## 7. Individual Agent Deep Dive

Select an agent to see their full timeline.

In [ ]:
target_agent = agents[0] if agents else "unknown"
timeline = build_agent_timeline(events, target_agent)

print(f"=== {target_agent} Full Timeline ({len(timeline)} events) ===")
for e in timeline:
    step = e.get("step", "?")
    etype = e.get("event_type", "?")
    data = e.get("data", {})
    if isinstance(data, dict):
        snippet = json.dumps(data, ensure_ascii=False)[:120]
    else:
        snippet = str(data)[:120]
    print(f"  Step {step:>4} [{etype:<14}] {snippet}")